# baseline_compare_metadata_effect 노트북 목표
1. 04번에서 학습한 제안 모델(`KcBERT PCA + Metadata + MLP`)을 baseline 모델과 같은 기준으로 비교한다.
2. baseline으로 `baseline_1` 텍스트 계열(TF-IDF + Random Forest, TF-IDF + Linear SVM), `baseline_2` Metadata only + MLP, `baseline_3` KcBERT PCA only + MLP를 비교한다.
3. 메타데이터 조합 7가지를 ablation 실험으로 비교해 어떤 메타데이터가 성능에 기여하는지 확인한다.
4. baseline 모델과 ablation best 모델을 저장한다.


## 1. 라이브러리 로드

- baseline 비교, MLP ablation, 성능 평가, 모델 저장에 필요한 패키지를 불러온다.
- 각 패키지의 역할은 다음과 같다.

1. `pandas` / `numpy`: 데이터프레임 처리와 수치 계산
2. `joblib` / `json`: 모델 bundle과 설정 파일 저장 및 로드
3. `TfidfVectorizer` / `RandomForestClassifier` / `LinearSVC` / `CalibratedClassifierCV`: 전통적 텍스트 baseline과 TF-IDF 보조 튜닝 구성
4. `PCA` / `StandardScaler`: KcBERT 임베딩 축소와 메타데이터 정규화
5. `SMOTE` / `ImbPipeline`: 클래스 불균형 보정이 포함된 MLP pipeline 구성
6. `sklearn.metrics`: accuracy, F1, PR-AUC 등 공통 평가 지표 계산


In [1]:
import json
from itertools import combinations

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC


## 2. 데이터 로드 및 split 재현

- 02번에서 생성한 `reviews_embeddings_extract.csv`를 불러온다.
- 04번과 같은 random seed와 stratify 조건으로 train / validation / test split을 재현한다.
- 텍스트 baseline에는 `cleaned_review_text`를 사용하고, MLP 계열 모델에는 KcBERT 임베딩과 메타데이터를 사용한다.


In [2]:
SEED = 42
LABEL_COL = 'label'
TEXT_COL = 'cleaned_review_text'
MIN_TUNED_THRESHOLD = 0.1

raw_df = pd.read_csv('csv/reviews_embeddings_extract.csv')

emb_cols = [f'kcbert_{i}' for i in range(768)]
meta_cols = ['text_length', 'cleaned_text_length', 'emoji_count', 'photo_count', 'has_emoji', 'has_photo', 'is_very_short_review']
hybrid_cols = emb_cols + meta_cols

X_all = raw_df[hybrid_cols].copy()
y_all = raw_df[LABEL_COL].astype(int)

X_train, X_temp, y_train, y_temp = train_test_split(
    X_all,
    y_all,
    test_size=0.3,
    random_state=SEED,
    stratify=y_all,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=SEED,
    stratify=y_temp,
)

train_idx = X_train.index
val_idx = X_val.index
test_idx = X_test.index

text_train = raw_df.loc[train_idx, TEXT_COL].fillna('').astype(str)
text_val = raw_df.loc[val_idx, TEXT_COL].fillna('').astype(str)
text_test = raw_df.loc[test_idx, TEXT_COL].fillna('').astype(str)

print('Raw embedding data:', raw_df.shape)
print('Hybrid train:', X_train.shape)
print('Hybrid validation:', X_val.shape)
print('Hybrid test:', X_test.shape)
print('KcBERT feature 수:', len(emb_cols))
print('metadata feature:', meta_cols)
print('excluded feature:', ['rating'])
print('train label 분포:', y_train.value_counts().sort_index().to_dict())
print('validation label 분포:', y_val.value_counts().sort_index().to_dict())
print('test label 분포:', y_test.value_counts().sort_index().to_dict())

Raw embedding data: (8841, 786)
Hybrid train: (6188, 775)
Hybrid validation: (1326, 775)
Hybrid test: (1327, 775)
KcBERT feature 수: 768
metadata feature: ['text_length', 'cleaned_text_length', 'emoji_count', 'photo_count', 'has_emoji', 'has_photo', 'is_very_short_review']
excluded feature: ['rating']
train label 분포: {0: 3983, 1: 2205}
validation label 분포: {0: 854, 1: 472}
test label 분포: {0: 854, 1: 473}


/tmp/ipykernel_60537/271080005.py:6: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv('csv/reviews_embeddings_extract.csv')


## 3. ?? ?? ?? ??

- ?? ??? ?? ???? ???? ?? threshold ??? ?? ?? ??? ???? ????.
- ??? ?? ? ??? ??? ??? ??? ????.
- ? ??? threshold ???? ??? ??(1), threshold ???? ?? ??(0)? ????.
- threshold ??? `0.1 ??`? ????, validation ????? F1-score? ?? ?? ?? ????.
- ?? ????? validation?? ??? threshold? validation/test? ???? ??? ??? ???.
- ?? ?? ? ?? ?? threshold ??, ?? ?? ??, ?? ? ??? ????.
- ? ?? ?? ?? ??? baseline? tabular baseline? ???? ???? helper ??? ????.

### ?? ?? ??
- Precision: ????? ??? ?? ? ?? ??? ?? ????.
- Recall: ?? ??? ?? ? ??? ??? ????.
- F1-score: Precision? Recall? ?? ????, ?? ??? 1?? ????.
- PR-AUC: ??? ????? Precision-Recall ??? ????? ?? ?? ???.
- Accuracy: ?? ?? ?? ?? ?? ????.
- ROC-AUC: ?? ??? ??? ??? ????? ??? ? ????? ?? ???.
- FP / FN: ?? ?? ??? ???? ?? ? ??, ??? ??? ?? ???.


In [3]:
def fit_probability_calibrator(y_true, raw_prob):
    clipped = np.clip(raw_prob, 1e-6, 1 - 1e-6)
    calibrator = IsotonicRegression(out_of_bounds='clip')
    calibrator.fit(clipped, y_true)
    return calibrator


def apply_probability_calibration(calibrator, raw_prob):
    clipped = np.clip(raw_prob, 1e-6, 1 - 1e-6)
    return np.clip(calibrator.predict(clipped), 0.0, 1.0)


def calibrate_validation_and_test_probabilities(y_val, y_test, val_prob_raw, test_prob_raw):
    calibrator = fit_probability_calibrator(y_val, val_prob_raw)
    val_prob = apply_probability_calibration(calibrator, val_prob_raw)
    test_prob = apply_probability_calibration(calibrator, test_prob_raw)
    calibration_summary = {
        'method': 'isotonic_regression',
        'validation_brier_raw': float(brier_score_loss(y_val, val_prob_raw)),
        'validation_brier_calibrated': float(brier_score_loss(y_val, val_prob)),
        'test_brier_raw': float(brier_score_loss(y_test, test_prob_raw)),
        'test_brier_calibrated': float(brier_score_loss(y_test, test_prob)),
    }
    return calibrator, val_prob, test_prob, calibration_summary


def tune_threshold_from_validation(y_true, prob, min_threshold=0.1):
    precisions, recalls, thresholds = precision_recall_curve(y_true, prob)

    if len(thresholds) == 0:
        return float(min_threshold), pd.DataFrame()

    f1s = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-8)
    candidates = pd.DataFrame({
        'threshold': thresholds,
        'precision': precisions[:-1],
        'recall': recalls[:-1],
        'f1': f1s,
    })
    candidates = candidates[candidates['threshold'] >= min_threshold]

    if candidates.empty:
        return float(min_threshold), candidates

    best_threshold = float(candidates.sort_values('f1', ascending=False).iloc[0]['threshold'])
    return best_threshold, candidates.sort_values('f1', ascending=False).reset_index(drop=True)


def metric_row(model_name, feature_set, split, y_true, prob, threshold):
    pred = (prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    return {
        'model': model_name,
        'feature_set': feature_set,
        'split': split,
        'threshold': float(threshold),
        'accuracy': float(accuracy_score(y_true, pred)),
        'f1': float(f1_score(y_true, pred)),
        'pr_auc': float(average_precision_score(y_true, prob)),
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'recall': float(recall_score(y_true, pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, prob)),
        'brier': float(brier_score_loss(y_true, prob)),
        'tn': int(cm[0, 0]),
        'fp': int(cm[0, 1]),
        'fn': int(cm[1, 0]),
        'tp': int(cm[1, 1]),
    }


def evaluate_probabilities(model_name, feature_set, y_val, val_prob, y_test, test_prob):
    best_threshold, threshold_candidates = tune_threshold_from_validation(
        y_val,
        val_prob,
        min_threshold=MIN_TUNED_THRESHOLD,
    )

    rows = [
        metric_row(model_name, feature_set, 'validation_tuned_min_0_1', y_val, val_prob, best_threshold),
        metric_row(model_name, feature_set, 'test_tuned_min_0_1', y_test, test_prob, best_threshold),
    ]
    return rows, best_threshold, threshold_candidates


def make_model_result_display(rows):
    display_df = pd.DataFrame(rows).copy()
    display_df = display_df[display_df['split'].isin(['validation_tuned_min_0_1', 'test_tuned_min_0_1'])]
    display_df['평가 데이터'] = display_df['split'].replace({'validation_tuned_min_0_1': 'Validation', 'test_tuned_min_0_1': 'Test'})
    display_df = display_df.rename(columns={
        'threshold': 'threshold',
        'f1': 'F1',
        'pr_auc': 'PR-AUC',
        'precision': 'Precision',
        'recall': 'Recall',
        'accuracy': 'Accuracy',
        'roc_auc': 'ROC-AUC',
        'brier': 'Brier',
        'fp': 'FP(일반→이벤트)',
        'fn': 'FN(이벤트→일반)',
        'tp': 'TP(이벤트→이벤트)',
        'tn': 'TN(일반→일반)',
    })
    return display_df[['평가 데이터', 'threshold', 'F1', 'PR-AUC', 'Precision', 'Recall', 'Accuracy', 'ROC-AUC', 'Brier', 'FP(일반→이벤트)', 'FN(이벤트→일반)', 'TP(이벤트→이벤트)', 'TN(일반→일반)']]


def display_model_result(rows, title):
    print(title)
    display(make_model_result_display(rows).round({'threshold': 4, 'F1': 4, 'PR-AUC': 4, 'Precision': 4, 'Recall': 4, 'Accuracy': 4, 'ROC-AUC': 4, 'Brier': 4}))


### 3-1. 텍스트 baseline 공통 helper

- `default_tfidf_params()`: 기본 TF-IDF 설정을 한 곳에서 관리한다.
- `make_text_pipeline()`: 분류기 종류에 따라 `TF-IDF + Random Forest` 또는 `TF-IDF + Linear SVM` pipeline을 만든다.
- `fit_text_baseline()`: train 학습, validation/test 확률 예측, threshold 평가, joblib 저장을 한 번에 처리한다.
- 이 helper를 통해 `baseline_1a`, `baseline_1b`가 같은 평가 규칙을 공유한다.


In [4]:
def default_tfidf_params():
    return {
        'max_features': 20000,
        'ngram_range': (1, 2),
        'min_df': 2,
        'max_df': 0.95,
        'sublinear_tf': True,
    }


def make_calibrated_linear_svm(random_state=SEED):
    base_svm = LinearSVC(C=1.0, class_weight='balanced', random_state=random_state)
    return CalibratedClassifierCV(base_svm, method='sigmoid', cv=5)


def make_text_pipeline(classifier_type, vectorizer_params=None, random_state=SEED):
    tfidf_params = default_tfidf_params()
    if vectorizer_params:
        tfidf_params.update(vectorizer_params)

    vectorizer = TfidfVectorizer(**tfidf_params)

    if classifier_type == 'rf':
        classifier = RandomForestClassifier(n_estimators=300, random_state=random_state, n_jobs=-1, class_weight='balanced')
        classifier_name = 'rf'
    elif classifier_type == 'svm':
        classifier = make_calibrated_linear_svm(random_state=random_state)
        classifier_name = 'svm'
    else:
        raise ValueError(f'지원하지 않는 classifier_type입니다: {classifier_type}')

    return Pipeline([('tfidf', vectorizer), (classifier_name, classifier)])


def fit_text_baseline(model_name, feature_set, classifier_type, vectorizer_params=None, save_path=None):
    model = make_text_pipeline(classifier_type=classifier_type, vectorizer_params=vectorizer_params, random_state=SEED)
    model.fit(text_train, y_train)

    val_prob_raw = model.predict_proba(text_val)[:, 1]
    test_prob_raw = model.predict_proba(text_test)[:, 1]
    probability_calibrator, val_prob, test_prob, calibration_summary = calibrate_validation_and_test_probabilities(y_val, y_test, val_prob_raw, test_prob_raw)

    rows, best_threshold, threshold_candidates = evaluate_probabilities(model_name, feature_set, y_val, val_prob, y_test, test_prob)

    if save_path is not None:
        joblib.dump({
            'model': model,
            'model_name': model_name,
            'text_col': TEXT_COL,
            'target_col': LABEL_COL,
            'best_threshold': best_threshold,
            'vectorizer_params': dict(default_tfidf_params(), **(vectorizer_params or {})),
            'classifier_type': classifier_type,
            'probability_calibrator': probability_calibrator,
            'probability_calibration': calibration_summary,
        }, save_path)

    return model, rows, best_threshold, threshold_candidates, probability_calibrator, calibration_summary


## 4. 제안 모델 결과 로드

- 04번에서 저장한 `proposed_mlp_final_model.joblib`과 `proposed_mlp_selected_config.json`을 불러온다.
- 이 모델은 `KcBERT PCA + Metadata + MLP` 구조의 프로젝트 제안 모델이다.
- 05번에서는 제안 모델을 다시 학습하지 않고, 저장된 모델로 validation/test 성능을 같은 평가 함수에 맞춰 계산한다.


In [5]:
with open('outputs/proposed_mlp_selected_config.json', 'r', encoding='utf-8') as f:
    proposed_config = json.load(f)

proposed_bundle_for_metrics = joblib.load('outputs/proposed_mlp_final_model.joblib')
proposed_model_for_metrics = proposed_bundle_for_metrics['model']
proposed_feature_cols = proposed_bundle_for_metrics['feature_cols']
proposed_threshold = float(proposed_bundle_for_metrics.get('best_threshold', proposed_config.get('best_threshold_from_validation', 0.5)))
proposed_calibrator = proposed_bundle_for_metrics.get('probability_calibrator')

proposed_val_prob_raw = proposed_model_for_metrics.predict_proba(X_val[proposed_feature_cols])[:, 1]
proposed_test_prob_raw = proposed_model_for_metrics.predict_proba(X_test[proposed_feature_cols])[:, 1]
if proposed_calibrator is not None:
    proposed_val_prob = apply_probability_calibration(proposed_calibrator, proposed_val_prob_raw)
    proposed_test_prob = apply_probability_calibration(proposed_calibrator, proposed_test_prob_raw)
else:
    proposed_val_prob = proposed_val_prob_raw
    proposed_test_prob = proposed_test_prob_raw

proposed_metrics = pd.DataFrame([
    metric_row('proposed_hybrid_mlp_04', 'kcbert_pca+metadata', 'validation_tuned_min_0_1', y_val, proposed_val_prob, proposed_threshold),
    metric_row('proposed_hybrid_mlp_04', 'kcbert_pca+metadata', 'test_tuned_min_0_1', y_test, proposed_test_prob, proposed_threshold),
])

display_model_result(proposed_metrics, '제안 모델(Hybrid MLP) 결과')


제안 모델(Hybrid MLP) 결과


,평가 데이터,threshold,F1,PR-AUC,Precision,Recall,Accuracy,ROC-AUC,Brier,FP(일반→이벤트),FN(이벤트→일반),TP(이벤트→이벤트),TN(일반→일반)
0,Validation,0.2667,0.5250,0.3920,0.3560,1.0,0.3560,0.5553,0.2272,854,0,472,0
1,Test,0.2667,0.5256,0.3991,0.3564,1.0,0.3564,0.5616,0.2267,854,0,473,0


### 5-1. TF-IDF 설정 점검

- 제안 모델을 바꾸지 않고, `baseline_1` 텍스트 계열 모델에 공통 적용할 TF-IDF 설정만 보조적으로 점검하는 단계이다.
- 같은 `cleaned_review_text`를 사용하되, 계산 부담이 비교적 작은 `TF-IDF + Linear SVM`으로 후보 설정을 먼저 비교한다.
- 여기서 고른 TF-IDF 설정은 뒤의 `baseline_1a`, `baseline_1b` 두 텍스트 baseline에 공통 적용한다.


In [6]:
tfidf_tuning_specs = [
    ('unigram_15000', {
        'max_features': 15000,
        'ngram_range': (1, 1),
        'min_df': 2,
        'max_df': 0.95,
        'sublinear_tf': False,
    }),
    ('uni_bigram_20000', {
        'max_features': 20000,
        'ngram_range': (1, 2),
        'min_df': 2,
        'max_df': 0.95,
        'sublinear_tf': False,
    }),
    ('uni_bigram_30000_sublinear', {
        'max_features': 30000,
        'ngram_range': (1, 2),
        'min_df': 2,
        'max_df': 0.95,
        'sublinear_tf': True,
    }),
    ('uni_bigram_min_df3', {
        'max_features': 20000,
        'ngram_range': (1, 2),
        'min_df': 3,
        'max_df': 0.95,
        'sublinear_tf': True,
    }),
]

tfidf_tuning_summary_rows = []

for spec_name, vectorizer_params in tfidf_tuning_specs:
    _, spec_rows, best_threshold, _, _, _ = fit_text_baseline(
        model_name=f'tfidf_tuning_linear_svm_{spec_name}',
        feature_set=f'tfidf_tuning:{spec_name}',
        classifier_type='svm',
        vectorizer_params=vectorizer_params,
        save_path=None,
    )

    spec_df = pd.DataFrame(spec_rows)
    val_row = spec_df.loc[spec_df['split'] == 'validation_tuned_min_0_1'].iloc[0]
    test_row = spec_df.loc[spec_df['split'] == 'test_tuned_min_0_1'].iloc[0]

    tfidf_tuning_summary_rows.append({
        'tfidf_name': spec_name,
        'best_threshold': best_threshold,
        'val_f1': val_row['f1'],
        'val_pr_auc': val_row['pr_auc'],
        'val_precision': val_row['precision'],
        'val_recall': val_row['recall'],
        'test_f1': test_row['f1'],
        'test_pr_auc': test_row['pr_auc'],
        'test_precision': test_row['precision'],
        'test_recall': test_row['recall'],
    })

tfidf_tuning_summary = pd.DataFrame(tfidf_tuning_summary_rows).sort_values(
    ['val_f1', 'val_pr_auc'],
    ascending=False,
).reset_index(drop=True)

selected_tfidf_name = tfidf_tuning_summary.iloc[0]['tfidf_name']
selected_tfidf_params = dict(dict(tfidf_tuning_specs)[selected_tfidf_name])

tfidf_tuning_summary.to_csv('outputs/tfidf_tuning_summary.csv', index=False, encoding='utf-8-sig')
display(tfidf_tuning_summary.round(4))
print(f'선택된 TF-IDF 설정: {selected_tfidf_name}')


,tfidf_name,best_threshold,val_f1,val_pr_auc,val_precision,val_recall,test_f1,test_pr_auc,test_precision,test_recall
0,unigram_15000,0.2667,0.5277,0.4280,0.3587,0.9979,0.5259,0.4286,0.3570,0.9979
1,uni_bigram_min_df3,0.2727,0.5263,0.4287,0.3577,0.9958,0.5240,0.4095,0.3561,0.9915
2,uni_bigram_20000,0.2778,0.5263,0.4325,0.3574,0.9979,0.5251,0.4233,0.3565,0.9958
3,uni_bigram_30000_sublinear,0.2937,0.5263,0.4314,0.3587,0.9873,0.5276,0.4219,0.3597,0.9894


선택된 TF-IDF 설정: unigram_15000


## 5. baseline_1: TF-IDF 텍스트 baseline 비교

- `baseline_1`은 `cleaned_review_text`만 사용하는 전통적 텍스트 baseline 묶음이다.
- 이번 단계에서는 `baseline_1a: TF-IDF + Random Forest`, `baseline_1b: TF-IDF + Linear SVM` 두 모델을 같은 조건에서 비교한다.
- 먼저 기존 구현과 같은 방식의 TF-IDF + Random Forest baseline을 재현하고, 이어서 보조 튜닝으로 고른 TF-IDF 설정을 두 모델에 공통 적용한다.
- 이렇게 하면 수행계획서에 있던 `TF-IDF + Random Forest`, `TF-IDF + SVM` baseline을 현재 파이프라인 안에서 함께 비교할 수 있다.


In [7]:
tfidf_rf = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2, max_df=0.95)),
    ('rf', RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1, class_weight='balanced')),
])

tfidf_rf.fit(text_train, y_train)

tfidf_val_prob_raw = tfidf_rf.predict_proba(text_val)[:, 1]
tfidf_test_prob_raw = tfidf_rf.predict_proba(text_test)[:, 1]
probability_calibrator, tfidf_val_prob, tfidf_test_prob, tfidf_calibration_summary = calibrate_validation_and_test_probabilities(y_val, y_test, tfidf_val_prob_raw, tfidf_test_prob_raw)

tfidf_rows, tfidf_best_threshold, tfidf_threshold_candidates = evaluate_probabilities('baseline_1_tfidf_random_forest', 'cleaned_text_tfidf', y_val, tfidf_val_prob, y_test, tfidf_test_prob)

joblib.dump({
    'model': tfidf_rf,
    'model_name': 'baseline_1_tfidf_random_forest',
    'text_col': TEXT_COL,
    'target_col': LABEL_COL,
    'best_threshold': tfidf_best_threshold,
    'probability_calibrator': probability_calibrator,
    'probability_calibration': tfidf_calibration_summary,
}, 'outputs/baseline_1_tfidf_random_forest_model.joblib')

display_model_result(tfidf_rows, 'TF-IDF + Random Forest 결과')


TF-IDF + Random Forest 결과


,평가 데이터,threshold,F1,PR-AUC,Precision,Recall,Accuracy,ROC-AUC,Brier,FP(일반→이벤트),FN(이벤트→일반),TP(이벤트→이벤트),TN(일반→일반)
0,Validation,0.2823,0.5250,0.4073,0.3560,1.0,0.3560,0.5656,0.2256,854,0,472,0
1,Test,0.2823,0.5256,0.4236,0.3564,1.0,0.3564,0.5827,0.2251,854,0,473,0


### 5-2. baseline_1a: TF-IDF + Random Forest 재적합

- 위 셀은 기존 구현을 그대로 재현한 결과이고, 아래 셀에서는 5-1에서 고른 TF-IDF 설정으로 `baseline_1_tfidf_random_forest`를 다시 학습한다.
- 이렇게 하면 01~08의 기본 흐름은 유지하면서도, 텍스트 baseline을 더 공정한 조건으로 비교할 수 있다.


In [8]:
tfidf_rf, tfidf_rows, tfidf_best_threshold, tfidf_threshold_candidates, tfidf_rf_calibrator, tfidf_rf_calibration_summary = fit_text_baseline(
    'baseline_1_tfidf_random_forest',
    'cleaned_text_tfidf_tuned',
    classifier_type='rf',
    vectorizer_params=selected_tfidf_params,
    save_path='outputs/baseline_1_tfidf_random_forest_model.joblib',
)

display_model_result(tfidf_rows, 'TF-IDF + Random Forest 재적합 결과')


TF-IDF + Random Forest 재적합 결과


,평가 데이터,threshold,F1,PR-AUC,Precision,Recall,Accuracy,ROC-AUC,Brier,FP(일반→이벤트),FN(이벤트→일반),TP(이벤트→이벤트),TN(일반→일반)
0,Validation,0.3209,0.5261,0.4089,0.3655,0.9386,0.3982,0.5740,0.2249,769,29,443,85
1,Test,0.3209,0.5203,0.4138,0.3624,0.9218,0.3941,0.5755,0.2248,767,37,436,87


### 5-3. baseline_1b: TF-IDF + Linear SVM 모델 저장

- 수행계획서에 있던 `TF-IDF + SVM` baseline을 현재 데이터셋과 분할 규칙에 맞춰 추가한다.
- 5-1에서 고른 TF-IDF 설정을 그대로 적용해 `Random Forest`와 `Linear SVM`을 같은 조건에서 비교한다.
- `LinearSVC`는 확률을 직접 출력하지 않으므로 `CalibratedClassifierCV`를 사용해 확률 형태로 보정한 뒤 같은 threshold 평가 함수를 적용한다.


In [9]:
tfidf_svm, tfidf_svm_rows, tfidf_svm_best_threshold, tfidf_svm_threshold_candidates, tfidf_svm_calibrator, tfidf_svm_calibration_summary = fit_text_baseline(
    'baseline_1_tfidf_linear_svm',
    'cleaned_text_tfidf_tuned',
    classifier_type='svm',
    vectorizer_params=selected_tfidf_params,
    save_path='outputs/baseline_1_tfidf_linear_svm_model.joblib',
)

display_model_result(tfidf_svm_rows, 'TF-IDF + Linear SVM 결과')


TF-IDF + Linear SVM 결과


,평가 데이터,threshold,F1,PR-AUC,Precision,Recall,Accuracy,ROC-AUC,Brier,FP(일반→이벤트),FN(이벤트→일반),TP(이벤트→이벤트),TN(일반→일반)
0,Validation,0.2667,0.5277,0.4280,0.3587,0.9979,0.3643,0.5870,0.2233,842,1,471,12
1,Test,0.2667,0.5259,0.4286,0.3570,0.9979,0.3587,0.5885,0.2241,850,1,472,4


## 6. baseline_2: Metadata only + MLP ??

- `baseline_2_metadata_only_mlp`? ???? ????.
- ?? feature? `text_length`, `cleaned_text_length`, `emoji_count`, `photo_count`, `has_emoji`, `has_photo`, `is_very_short_review`? ????.
- KcBERT ??? ?? ?? ???????? ??? ??? ??? ??? ? ??? ????.
- ?? ?? ? ?? ?? metadata-only, KcBERT-only, ablation ??? ?? ???? ?? MLP helper? ????.
- ? ?? ?? ??? `baseline_2_metadata_only_mlp`? ??? ???? ????.


### 6-1. MLP baseline 공통 helper

- `make_mlp_model()`: KcBERT 임베딩 사용 여부와 메타데이터 컬럼 조합에 따라 MLP pipeline을 만든다.
- `fit_mlp_variant()`: 전달된 feature 조합으로 모델을 학습하고 validation/test 성능과 threshold를 계산한다.
- 이 helper를 재사용해 `baseline_2`, `baseline_3`, metadata ablation 모델들을 같은 조건으로 비교한다.


In [10]:
def make_mlp_model(cols, use_emb, metadata_cols=None, random_state=SEED):
    metadata_cols = list(metadata_cols or [])
    transformers = []
    if use_emb:
        transformers.append(('kcbert_pca', PCA(n_components=proposed_config.get('pca_n_components', 0.90), random_state=random_state), emb_cols))
    if metadata_cols:
        transformers.append(('metadata_scaler', StandardScaler(), metadata_cols))

    preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')

    return ImbPipeline([
        ('preprocess', preprocessor),
        ('smote', SMOTE(random_state=random_state, k_neighbors=proposed_config.get('smote_k_neighbors', 5))),
        ('mlp', MLPClassifier(
            hidden_layer_sizes=tuple(proposed_config['hidden_layer_sizes']),
            activation=proposed_config.get('activation', 'relu'),
            solver=proposed_config.get('solver', 'adam'),
            alpha=proposed_config['alpha'],
            batch_size=proposed_config.get('batch_size', 64),
            learning_rate_init=proposed_config.get('learning_rate_init', 1e-3),
            max_iter=200,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=10,
            random_state=random_state,
        )),
    ])


def fit_mlp_variant(model_name, feature_set, cols, use_emb, metadata_cols=None):
    cols = list(cols)
    metadata_cols = list(metadata_cols or [])
    X_train_variant = X_train[cols]
    X_val_variant = X_val[cols]
    X_test_variant = X_test[cols]

    model = make_mlp_model(cols, use_emb=use_emb, metadata_cols=metadata_cols, random_state=SEED)
    model.fit(X_train_variant, y_train)

    val_prob_raw = model.predict_proba(X_val_variant)[:, 1]
    test_prob_raw = model.predict_proba(X_test_variant)[:, 1]
    probability_calibrator, val_prob, test_prob, calibration_summary = calibrate_validation_and_test_probabilities(y_val, y_test, val_prob_raw, test_prob_raw)

    rows, best_threshold, threshold_candidates = evaluate_probabilities(model_name, feature_set, y_val, val_prob, y_test, test_prob)
    return model, rows, best_threshold, threshold_candidates, probability_calibrator, calibration_summary


In [11]:
metadata_only_model, metadata_only_rows, metadata_only_threshold, metadata_only_candidates, metadata_only_calibrator, metadata_only_calibration_summary = fit_mlp_variant(
    'baseline_2_metadata_only_mlp',
    'metadata_only',
    meta_cols,
    use_emb=False,
    metadata_cols=meta_cols,
)

joblib.dump({
    'model': metadata_only_model,
    'model_name': 'baseline_2_metadata_only_mlp',
    'feature_cols': meta_cols,
    'emb_cols': [],
    'meta_cols': meta_cols,
    'target_col': LABEL_COL,
    'input_type': 'tabular_raw',
    'best_threshold': metadata_only_threshold,
    'selected_config': proposed_config,
    'probability_calibrator': metadata_only_calibrator,
    'probability_calibration': metadata_only_calibration_summary,
}, 'outputs/baseline_2_metadata_only_mlp_model.joblib')

display_model_result(metadata_only_rows, 'Metadata only + MLP 결과')


Metadata only + MLP 결과


,평가 데이터,threshold,F1,PR-AUC,Precision,Recall,Accuracy,ROC-AUC,Brier,FP(일반→이벤트),FN(이벤트→일반),TP(이벤트→이벤트),TN(일반→일반)
0,Validation,0.2947,0.5285,0.3938,0.3617,0.9809,0.3771,0.5629,0.2258,817,9,463,37
1,Test,0.2947,0.5214,0.3688,0.3557,0.9767,0.3610,0.5219,0.2312,837,11,462,17


## 7. baseline_3: KcBERT only + MLP 모델 저장

- `baseline_3_kcbert_only_mlp`에 해당하는 모델이다.
- 입력 feature는 KcBERT 임베딩 768차원만 사용한다.
- 04번에서 선택된 MLP 설정을 그대로 사용해, 제안 모델에서 메타데이터를 뺐을 때 성능이 어떻게 달라지는지 확인한다.


In [12]:
pca_only_model, pca_only_rows, pca_only_threshold, pca_only_candidates, pca_only_calibrator, pca_only_calibration_summary = fit_mlp_variant(
    'baseline_3_kcbert_only_mlp',
    'kcbert_pca_only',
    emb_cols,
    use_emb=True,
    metadata_cols=[],
)

joblib.dump({
    'model': pca_only_model,
    'model_name': 'baseline_3_kcbert_only_mlp',
    'feature_cols': emb_cols,
    'emb_cols': emb_cols,
    'meta_cols': [],
    'target_col': LABEL_COL,
    'input_type': 'tabular_raw',
    'best_threshold': pca_only_threshold,
    'selected_config': proposed_config,
    'probability_calibrator': pca_only_calibrator,
    'probability_calibration': pca_only_calibration_summary,
}, 'outputs/baseline_3_kcbert_only_mlp_model.joblib')

display_model_result(pca_only_rows, 'KcBERT only + MLP 결과')


KcBERT only + MLP 결과


,평가 데이터,threshold,F1,PR-AUC,Precision,Recall,Accuracy,ROC-AUC,Brier,FP(일반→이벤트),FN(이벤트→일반),TP(이벤트→이벤트),TN(일반→일반)
0,Validation,0.328,0.5250,0.3767,0.3560,1.0,0.3560,0.5343,0.2281,854,0,472,0
1,Test,0.328,0.5256,0.3954,0.3564,1.0,0.3564,0.5547,0.2271,854,0,473,0


## 8. ????? ?? ?? ? ablation ??

- ablation? feature? ??? ??? ??? ??? ? ?? ??? ??? ????? ???? ????.
- ?? ??? ?????? `text_length`, `cleaned_text_length`, `emoji_count`, `photo_count`, `has_emoji`, `has_photo`, `is_very_short_review` 7??.
- 7? feature? ??? ?? ?? ?? ?? ??? ??? ?? MLP ???? ????.

??? ??? ? 127??.
- ?? ?? ?? ??? 127? ?? ??? ???, ? ???? ?? helper? ?? ????.
- ?? ??? ?? train/validation/test split, ?? MLP ??, ?? threshold ?? ???? ????.
- ?? ??? validation F1-score? ???? ??, PR-AUC? ?? ???? ????.
- 127? ?? ??? ?? ??? ???? ??, validation F1 ?? ?? ?? 1?? ????.


In [13]:
metadata_ablation_specs = []
for size in range(1, len(meta_cols) + 1):
    for feature_tuple in combinations(meta_cols, size):
        feature_cols = list(feature_tuple)
        feature_slug = '_'.join(feature_cols)
        feature_label = '+'.join(feature_cols)
        metadata_ablation_specs.append({'model': f'ablation_metadata_{feature_slug}', 'feature_set': f'ablation_metadata:{feature_label}', 'feature_cols': feature_cols, 'input_type': 'tabular'})

metadata_ablation_rows = []
metadata_ablation_models = {}
metadata_ablation_thresholds = {}
metadata_ablation_calibrators = {}
metadata_ablation_calibration_summaries = {}
metadata_ablation_spec_by_model = {}

for spec in metadata_ablation_specs:
    model, rows, threshold, threshold_candidates, probability_calibrator, calibration_summary = fit_mlp_variant(spec['model'], spec['feature_set'], spec['feature_cols'], use_emb=False, metadata_cols=spec['feature_cols'])
    metadata_ablation_rows.extend(rows)
    metadata_ablation_models[spec['model']] = model
    metadata_ablation_thresholds[spec['model']] = threshold
    metadata_ablation_calibrators[spec['model']] = probability_calibrator
    metadata_ablation_calibration_summaries[spec['model']] = calibration_summary
    metadata_ablation_spec_by_model[spec['model']] = spec

metadata_ablation_metrics = pd.DataFrame(metadata_ablation_rows)
SELECTION_SPLIT = 'validation_tuned_min_0_1'
metadata_ablation_validation = (metadata_ablation_metrics[metadata_ablation_metrics['split'] == SELECTION_SPLIT].sort_values(['f1', 'pr_auc'], ascending=False).reset_index(drop=True))
metadata_ablation_validation.insert(0, 'rank', metadata_ablation_validation.index + 1)


def readable_feature_set(feature_set: str) -> str:
    return feature_set.replace('ablation_metadata:', '').replace('+', ' + ')


metadata_ablation_display = metadata_ablation_validation.copy()
metadata_ablation_display['메타데이터 조합'] = metadata_ablation_display['feature_set'].apply(readable_feature_set)
metadata_ablation_display['feature 수'] = metadata_ablation_display['메타데이터 조합'].apply(lambda value: len(value.split(' + ')))
metadata_ablation_display = metadata_ablation_display.rename(columns={'rank': '순위', 'threshold': 'threshold', 'f1': 'F1', 'pr_auc': 'PR-AUC', 'precision': 'Precision', 'recall': 'Recall', 'accuracy': 'Accuracy', 'roc_auc': 'ROC-AUC'})[['순위', '메타데이터 조합', 'feature 수', 'threshold', 'F1', 'PR-AUC', 'Precision', 'Recall', 'Accuracy', 'ROC-AUC']]

best_row = metadata_ablation_validation.iloc[0]
worst_row = metadata_ablation_validation.iloc[-1]
best_metadata_model_name = best_row['model']
best_metadata_spec = metadata_ablation_spec_by_model[best_metadata_model_name]
metadata_ablation_best_path = 'outputs/ablation_metadata_best_model.joblib'
metadata_ablation_registry = [{'model': best_metadata_spec['model'], 'feature_set': best_metadata_spec['feature_set'], 'path': metadata_ablation_best_path, 'input_type': best_metadata_spec['input_type']}]

joblib.dump({
    'model': metadata_ablation_models[best_metadata_model_name],
    'model_name': best_metadata_spec['model'],
    'feature_cols': best_metadata_spec['feature_cols'],
    'emb_cols': [],
    'meta_cols': best_metadata_spec['feature_cols'],
    'target_col': LABEL_COL,
    'input_type': 'tabular_raw',
    'best_threshold': metadata_ablation_thresholds[best_metadata_model_name],
    'selected_config': proposed_config,
    'metadata_ablation': True,
    'metadata_ablation_best': True,
    'probability_calibrator': metadata_ablation_calibrators[best_metadata_model_name],
    'probability_calibration': metadata_ablation_calibration_summaries[best_metadata_model_name],
}, metadata_ablation_best_path)

round_cols = {'threshold': 4, 'F1': 4, 'PR-AUC': 4, 'Precision': 4, 'Recall': 4, 'Accuracy': 4, 'ROC-AUC': 4}


## 9. 메타데이터 영향도 분석

- 04번 제안 모델에서 메타데이터 컬럼을 하나씩 섞어 permutation importance를 계산한다.
- 특정 메타데이터를 섞었을 때 F1-score가 크게 떨어지면, 해당 feature가 모델 예측에 더 민감하게 작동한 것으로 해석한다.
- 이 단계는 모델 선택 기준이 아니라 사후 해석용이다.
- permutation importance (순열 중요도): 특정 feature 값을 무작위로 섞어 성능이 얼마나 떨어지는지 보는 중요도 분석 방법이다.
- 바로 아래 코드 셀은 제안 모델의 validation 데이터 기준으로 메타데이터 중요도와 요약 표를 만든다.


In [14]:
def metadata_permutation_importance(model, X_val, y_val, metadata_cols, threshold, repeats=50, random_state=SEED):
    rng = np.random.default_rng(random_state)
    base_prob = model.predict_proba(X_val)[:, 1]
    base_pred = (base_prob >= threshold).astype(int)
    base_f1 = f1_score(y_val, base_pred)

    rows = []
    for col in metadata_cols:
        drops = []
        for _ in range(repeats):
            X_perm = X_val.copy()
            shuffled = X_perm[col].to_numpy().copy()
            rng.shuffle(shuffled)
            X_perm[col] = shuffled

            perm_prob = model.predict_proba(X_perm)[:, 1]
            perm_pred = (perm_prob >= threshold).astype(int)
            perm_f1 = f1_score(y_val, perm_pred)
            drops.append(base_f1 - perm_f1)

        rows.append({
            'metadata': col,
            'base_f1': float(base_f1),
            'mean_f1_drop': float(np.mean(drops)),
            'std_f1_drop': float(np.std(drops)),
            'repeats': repeats,
        })

    return pd.DataFrame(rows).sort_values('mean_f1_drop', ascending=False).reset_index(drop=True)


proposed_bundle = joblib.load('outputs/proposed_mlp_final_model.joblib')
proposed_model = proposed_bundle['model']
proposed_feature_cols = proposed_bundle['feature_cols']
proposed_threshold = proposed_bundle.get('best_threshold', proposed_config['best_threshold_from_validation'])

metadata_importance = metadata_permutation_importance(
    proposed_model,
    X_val[proposed_feature_cols],
    y_val,
    meta_cols,
    proposed_threshold,
    repeats=50,
)

metadata_importance_display = metadata_importance.copy()
metadata_importance_display.insert(0, 'rank', metadata_importance_display.index + 1)
metadata_importance_display = metadata_importance_display.rename(columns={
    'metadata': '메타데이터',
    'base_f1': '기준 F1',
    'mean_f1_drop': '평균 F1 감소폭',
    'std_f1_drop': 'F1 감소폭 표준편차',
    'repeats': '반복 횟수',
})
metadata_importance_display = metadata_importance_display.round({
    '기준 F1': 4,
    '평균 F1 감소폭': 4,
    'F1 감소폭 표준편차': 4,
})

top_metadata = metadata_importance.iloc[0]
print(
    f"가장 큰 영향을 준 메타데이터: {top_metadata['metadata']} "
    f"(평균 F1 감소폭: {top_metadata['mean_f1_drop']:.4f})"
)
display(metadata_importance_display)

가장 큰 영향을 준 메타데이터: has_photo (평균 F1 감소폭: 0.0054)


,rank,메타데이터,기준 F1,평균 F1 감소폭,F1 감소폭 표준편차,반복 횟수
0,1,has_photo,0.5182,0.0054,0.0029,50
1,2,has_emoji,0.5182,0.0011,0.0015,50
2,3,cleaned_text_length,0.5182,0.0007,0.0017,50
3,4,photo_count,0.5182,-0.0001,0.0014,50
4,5,emoji_count,0.5182,-0.0002,0.0008,50
5,6,is_very_short_review,0.5182,-0.0004,0.0011,50
6,7,text_length,0.5182,-0.0004,0.0016,50


## 10. 전체 비교표 출력 및 모델 저장

- 04번 제안 모델, `baseline_1` 텍스트 baseline 2종, KcBERT-only, metadata-only, metadata 조합 ablation 결과를 하나의 성능표로 합친다.
- 성능표는 노트북 화면에서 확인하고, 불필요한 CSV 파일로는 저장하지 않는다.
- 화면에는 각 모델 그룹에서 validation F1 기준 최고 결과만 요약해서 보여준다.
- 최종 선택 기준은 validation F1-score 우선, PR-AUC 보조 기준이다.
- test 결과는 선택된 모델의 참고 성능으로만 함께 표시한다.
- 06번이 후보 모델을 자동으로 읽을 수 있도록 `baseline_model_registry.csv`만 저장한다.
- metadata 조합 ablation은 31개를 모두 비교하지만, best 조합이 baseline_2와 같은 전체 메타데이터 조합이면 registry와 최종 그룹 요약에서는 baseline_2를 대표로 사용한다.
- 이 경우 ablation best는 별도 참고 메시지로만 출력해, 같은 feature set을 쓰는 모델이 최종 순위표에 중복으로 들어가지 않게 한다.
- 바로 아래 마지막 코드 셀은 전체 비교표 생성, registry 저장, 그룹별 최고 모델 요약을 한 번에 수행한다.


In [15]:
baseline_metrics = pd.concat([
    proposed_metrics,
    pd.DataFrame(tfidf_rows),
    pd.DataFrame(tfidf_svm_rows),
    pd.DataFrame(pca_only_rows),
    pd.DataFrame(metadata_only_rows),
    metadata_ablation_metrics,
], ignore_index=True)

validation_summary = (
    baseline_metrics[baseline_metrics['split'] == 'validation_tuned_min_0_1']
    .sort_values(['f1', 'pr_auc'], ascending=False)
    .reset_index(drop=True)
)
validation_summary.insert(0, 'rank', validation_summary.index + 1)

test_summary = (
    baseline_metrics[baseline_metrics['split'] == 'test_tuned_min_0_1']
    .sort_values(['f1', 'pr_auc'], ascending=False)
    .reset_index(drop=True)
)
test_summary.insert(0, 'rank', test_summary.index + 1)

validation_accuracy_summary = (
    baseline_metrics[baseline_metrics['split'] == 'validation_tuned_min_0_1']
    .sort_values(['accuracy', 'f1', 'pr_auc'], ascending=False)
    .reset_index(drop=True)
)
validation_accuracy_summary.insert(0, 'rank_accuracy', validation_accuracy_summary.index + 1)


metadata_ablation_duplicate_of_baseline_2 = (
    list(best_metadata_spec['feature_cols']) == list(meta_cols)
)
metadata_ablation_registry_for_selection = [] if metadata_ablation_duplicate_of_baseline_2 else metadata_ablation_registry

saved_model_files = pd.DataFrame([
    {'model': 'baseline_1_tfidf_random_forest', 'feature_set': 'cleaned_text_tfidf_tuned', 'path': 'outputs/baseline_1_tfidf_random_forest_model.joblib', 'input_type': 'text'},
    {'model': 'baseline_1_tfidf_linear_svm', 'feature_set': 'cleaned_text_tfidf_tuned', 'path': 'outputs/baseline_1_tfidf_linear_svm_model.joblib', 'input_type': 'text'},
    {'model': 'baseline_2_metadata_only_mlp', 'feature_set': 'metadata_only', 'path': 'outputs/baseline_2_metadata_only_mlp_model.joblib', 'input_type': 'tabular'},
    {'model': 'baseline_3_kcbert_only_mlp', 'feature_set': 'kcbert_pca_only', 'path': 'outputs/baseline_3_kcbert_only_mlp_model.joblib', 'input_type': 'tabular'},
    *metadata_ablation_registry_for_selection,
    {'model': 'proposed_hybrid_mlp_04', 'feature_set': 'kcbert_pca+metadata', 'path': 'outputs/proposed_mlp_final_model.joblib', 'input_type': 'tabular'},
])
saved_model_files.to_csv('outputs/baseline_model_registry.csv', index=False, encoding='utf-8-sig')

validation_selection = baseline_metrics[baseline_metrics['split'] == 'validation_tuned_min_0_1'].copy()
test_reference = baseline_metrics[baseline_metrics['split'] == 'test_tuned_min_0_1'].copy()

def readable_feature_label(feature_set):
    if feature_set.startswith('ablation_metadata:'):
        return readable_feature_set(feature_set)
    return {
        'cleaned_text_tfidf': 'cleaned_review_text TF-IDF',
        'cleaned_text_tfidf_tuned': f'cleaned_review_text TF-IDF tuned ({selected_tfidf_name})',
        'kcbert_pca_only': 'KcBERT PCA only',
        'metadata_only': 'text_length + emoji_count + photo_count',
        'kcbert_pca+metadata': 'KcBERT PCA + metadata',
    }.get(feature_set, feature_set)

model_groups = [
    ('baseline_1a: TF-IDF + Random Forest', validation_selection['model'].eq('baseline_1_tfidf_random_forest')),
    ('baseline_1b: TF-IDF + Linear SVM', validation_selection['model'].eq('baseline_1_tfidf_linear_svm')),
    ('baseline_2: Metadata only + MLP', validation_selection['model'].eq('baseline_2_metadata_only_mlp')),
    ('baseline_3: KcBERT only + MLP', validation_selection['model'].eq('baseline_3_kcbert_only_mlp')),
    ('제안 모델(Hybrid)', validation_selection['model'].eq('proposed_hybrid_mlp_04')),
]

if not metadata_ablation_duplicate_of_baseline_2:
    model_groups.append(
        ('ablation_metadata best', validation_selection['model'].str.startswith('ablation_metadata_'))
    )

best_rows = []
for group_name, mask in model_groups:
    group_candidates = validation_selection[mask]
    if group_candidates.empty:
        continue

    best_val = group_candidates.sort_values(['f1', 'pr_auc'], ascending=False).iloc[0]
    matching_test = test_reference[test_reference['model'] == best_val['model']]
    best_test = matching_test.iloc[0] if not matching_test.empty else None

    best_rows.append({
        '그룹': group_name,
        '선택 모델': best_val['model'],
        '입력 feature': readable_feature_label(best_val['feature_set']),
        'threshold': best_val['threshold'],
        'Validation F1': best_val['f1'],
        'Validation PR-AUC': best_val['pr_auc'],
        'Validation Precision': best_val['precision'],
        'Validation Recall': best_val['recall'],
        'Validation Accuracy': best_val['accuracy'],
        'Validation ROC-AUC': best_val['roc_auc'],
        'Test F1': np.nan if best_test is None else best_test['f1'],
        'Test PR-AUC': np.nan if best_test is None else best_test['pr_auc'],
        'Test Precision': np.nan if best_test is None else best_test['precision'],
        'Test Recall': np.nan if best_test is None else best_test['recall'],
        'Test Accuracy': np.nan if best_test is None else best_test['accuracy'],
        'Test ROC-AUC': np.nan if best_test is None else best_test['roc_auc'],
    })

best_by_group_summary = (
    pd.DataFrame(best_rows)
    .sort_values(['Validation F1', 'Validation PR-AUC'], ascending=False)
    .reset_index(drop=True)
)
best_by_group_summary.insert(0, '순위', best_by_group_summary.index + 1)
representative_validation_summary = validation_summary.copy()
if metadata_ablation_duplicate_of_baseline_2:
    representative_validation_summary = representative_validation_summary[
        ~representative_validation_summary['model'].str.startswith('ablation_metadata_')
    ].reset_index(drop=True)
project_best = representative_validation_summary.iloc[0]

round_cols = {
    'threshold': 4,
    'Validation F1': 4,
    'Validation PR-AUC': 4,
    'Validation Precision': 4,
    'Validation Recall': 4,
    'Validation Accuracy': 4,
    'Validation ROC-AUC': 4,
    'Test F1': 4,
    'Test PR-AUC': 4,
    'Test Precision': 4,
    'Test Recall': 4,
    'Test Accuracy': 4,
    'Test ROC-AUC': 4,
}

print('그룹별 최고 모델 요약')
print('기준: validation_tuned_min_0_1, F1 우선 + PR-AUC 보조')
display(best_by_group_summary.round(round_cols))

if metadata_ablation_duplicate_of_baseline_2:
    print('metadata ablation best는 baseline_2와 같은 전체 메타데이터 조합이므로 최종 그룹 요약과 registry에서는 중복 제외합니다.')
    display(pd.DataFrame([{
        '해석': '조합 실험 확인용',
        'ablation best 모델': best_metadata_model_name,
        '대표 baseline': 'baseline_2_metadata_only_mlp',
        '입력 feature': readable_feature_label(best_row['feature_set']),
        'Validation F1': best_row['f1'],
        'Validation PR-AUC': best_row['pr_auc'],
    }]).round({
        'Validation F1': 4,
        'Validation PR-AUC': 4,
    }))

print('전체 후보 중 validation F1 기준 1위')
display(pd.DataFrame([{
    '선택 모델': project_best['model'],
    '입력 feature': readable_feature_label(project_best['feature_set']),
    'Validation F1': project_best['f1'],
    'Validation PR-AUC': project_best['pr_auc'],
    'Validation Precision': project_best['precision'],
    'Validation Recall': project_best['recall'],
    'Validation Accuracy': project_best['accuracy'],
    'Validation ROC-AUC': project_best['roc_auc'],
}]).round({
    'Validation F1': 4,
    'Validation PR-AUC': 4,
    'Validation Precision': 4,
    'Validation Recall': 4,
    'Validation Accuracy': 4,
    'Validation ROC-AUC': 4,
}))

print(f"저장된 모델 registry: outputs/baseline_model_registry.csv ({len(saved_model_files)}개 모델)")

baseline_vs_proposed = best_by_group_summary[
    best_by_group_summary['그룹'].isin([
        'baseline_1: TF-IDF + Random Forest',
        'baseline_2: Metadata only + MLP',
        'baseline_3: KcBERT only + MLP',
        '제안 모델(Hybrid)',
    ])
].reset_index(drop=True)

baseline_vs_proposed = (
    baseline_vs_proposed[[
        '그룹',
        '선택 모델',
        '입력 feature',
        'threshold',
        'Validation F1',
        'Validation PR-AUC',
        'Validation Precision',
        'Validation Recall',
        'Test F1',
        'Test PR-AUC',
    ]]
    .sort_values(['Validation F1', 'Validation PR-AUC'], ascending=False)
    .reset_index(drop=True)
)
baseline_vs_proposed.insert(0, '비교 순위', baseline_vs_proposed.index + 1)

print('베이스라인 3개 vs 제안 모델 비교')
display(baseline_vs_proposed.round(round_cols))


그룹별 최고 모델 요약
기준: validation_tuned_min_0_1, F1 우선 + PR-AUC 보조


,순위,그룹,선택 모델,입력 feature,threshold,Validation F1,Validation PR-AUC,Validation Precision,Validation Recall,Validation Accuracy,Validation ROC-AUC,Test F1,Test PR-AUC,Test Precision,Test Recall,Test Accuracy,Test ROC-AUC
0,1,ablation_metadata best,ablation_metadata_text_length_cleaned_text_len...,text_length + cleaned_text_length + emoji_coun...,0.2896,0.5351,0.3935,0.3685,0.9767,0.3959,0.5692,0.5202,0.3911,0.3577,0.9535,0.3730,0.5518
1,2,baseline_2: Metadata only + MLP,baseline_2_metadata_only_mlp,text_length + emoji_count + photo_count,0.2947,0.5285,0.3938,0.3617,0.9809,0.3771,0.5629,0.5214,0.3688,0.3557,0.9767,0.3610,0.5219
2,3,baseline_1b: TF-IDF + Linear SVM,baseline_1_tfidf_linear_svm,cleaned_review_text TF-IDF tuned (unigram_15000),0.2667,0.5277,0.4280,0.3587,0.9979,0.3643,0.5870,0.5259,0.4286,0.3570,0.9979,0.3587,0.5885
3,4,baseline_1a: TF-IDF + Random Forest,baseline_1_tfidf_random_forest,cleaned_review_text TF-IDF tuned (unigram_15000),0.3209,0.5261,0.4089,0.3655,0.9386,0.3982,0.5740,0.5203,0.4138,0.3624,0.9218,0.3941,0.5755
4,5,제안 모델(Hybrid),proposed_hybrid_mlp_04,KcBERT PCA + metadata,0.2667,0.5250,0.3920,0.3560,1.0000,0.3560,0.5553,0.5256,0.3991,0.3564,1.0000,0.3564,0.5616
5,6,baseline_3: KcBERT only + MLP,baseline_3_kcbert_only_mlp,KcBERT PCA only,0.3280,0.5250,0.3767,0.3560,1.0000,0.3560,0.5343,0.5256,0.3954,0.3564,1.0000,0.3564,0.5547


전체 후보 중 validation F1 기준 1위


,선택 모델,입력 feature,Validation F1,Validation PR-AUC,Validation Precision,Validation Recall,Validation Accuracy,Validation ROC-AUC
0,ablation_metadata_text_length_cleaned_text_len...,text_length + cleaned_text_length + emoji_coun...,0.5351,0.3935,0.3685,0.9767,0.3959,0.5692


저장된 모델 registry: outputs/baseline_model_registry.csv (6개 모델)
베이스라인 3개 vs 제안 모델 비교


,비교 순위,그룹,선택 모델,입력 feature,threshold,Validation F1,Validation PR-AUC,Validation Precision,Validation Recall,Test F1,Test PR-AUC
0,1,baseline_2: Metadata only + MLP,baseline_2_metadata_only_mlp,text_length + emoji_count + photo_count,0.2947,0.5285,0.3938,0.3617,0.9809,0.5214,0.3688
1,2,제안 모델(Hybrid),proposed_hybrid_mlp_04,KcBERT PCA + metadata,0.2667,0.5250,0.3920,0.3560,1.0000,0.5256,0.3991
2,3,baseline_3: KcBERT only + MLP,baseline_3_kcbert_only_mlp,KcBERT PCA only,0.3280,0.5250,0.3767,0.3560,1.0000,0.5256,0.3954
